# Remove the outliers of features
by using a centered (!) moving median

In [ ]:
import math
from functools import partial
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas import Timedelta, DataFrame

from config import PATHS
from config.intervals import SEGMENT
from cycle_extraction.plot_features import plot_features
from feature_extraction.extract_features import FeatureNames

In [ ]:
def rolling_average(
        df: DataFrame,
        window: Timedelta,
        min_valid_duration: Timedelta,
        sampling_period: Timedelta,
        agg: Literal["mean", "median"],
        on: str = 'start_mtz',
) -> DataFrame:
    """
    Compute a centered time-based rolling aggregation for segment features while
    preserving original NaN rows.

    Parameters
    ----------
    df : DataFrame
        Input dataframe containing `start_mtz` (datetime) and feature columns.
        Aggregation is performed with `on='start_mtz'`.
    window : Timedelta
        Rolling window length (e.g. `Timedelta(minutes=10)`).
    min_valid_duration : Timedelta
        Minimum required valid duration inside each window for emitting a value.
        This is converted to `min_periods` as:
        `ceil(min_valid_duration / sampling_period)`.
    sampling_period : Timedelta
        Segment spacing (time between consecutive samples), e.g. 15 seconds.
    agg : {"mean", "median"}
        Aggregation function used on each window.
    on : str, default "start_mtz"
        Which column to use as index

    Returns
    -------
    DataFrame
        Rolling-aggregated dataframe with the same row count as `df`.
        Rows that were NaN in the input are forced back to NaN in the output
        to avoid implicit gap filling at this stage.

    Notes
    -----
    - `min_periods` counts non-NaN samples, even for time-based windows.
    - `center=True` and `closed='both'` are used for symmetric windows.

    Example
    -------
    segs_cleaned = rolling_average(
        segs,
        window=Timedelta(minutes=10),
        min_valid_duration=Timedelta(minutes=3),
        sampling_period=Timedelta(seconds=15),
        agg="median")
    """

    # min_periods: how many samples are required in a window for a value to be output
    min_periods = math.ceil(min_valid_duration / sampling_period)

    rolling = df.rolling(
        window=window,
        min_periods=min_periods,
        center=True,
        on=on,
        closed='both',
    )

    averages = getattr(rolling, agg)()

    # Make NaN rows stay NaN, rather than being filled with surrounding values
    na_rows = df.isna().any(axis='columns')
    averages.loc[na_rows, :] = pd.NA

    return averages


In [ ]:
rolling_average_seg = partial(rolling_average, sampling_period=SEGMENT.exact_dur)
feat_names = FeatureNames.ALL_ORDERED
plot_features = partial(plot_features, feat_names=feat_names)

pdir = PATHS.patient_dirs()[6]
print(pdir.name)
segs_full = pd.read_pickle(pdir.segments_table.pickle).drop(columns=['lead_szr', 'type', 'exists', 'start_index'])
segs = segs_full[['start_mtz', *feat_names]]
segs.head(10)

In [ ]:
# Raw Features
plot_features(segs).show()

In [ ]:
segs_above_thresh = segs_full[segs['var_P'] > 40000]
print(len(segs_above_thresh['var_P']) / len(segs) * 100, "% of segments above threshold")
segs_above_thresh

In [ ]:
segs_below_thresh = segs[segs['var_P'] < 20000]
# plot_features(segs_below_thresh).show()

In [ ]:
segs_averaged = rolling_average_seg(segs, window=Timedelta(minutes=10), min_valid_duration=Timedelta(minutes=0),
                                    agg='median')
plot_features(segs_averaged).show()

In [ ]:
extreme_segs = segs_full[segs_averaged['var_P'] > 20000]
print(f"From {extreme_segs['start_mtz'].min()} to {extreme_segs['start_mtz'].max()}")
extreme_segs

In [ ]:
first_extreme_idx = extreme_segs.index[0]
segs_full.loc[range(first_extreme_idx - 40, first_extreme_idx + 100)]

In [ ]:
plot_range = (first_extreme_idx - 100, first_extreme_idx + 100)
plot_features(segs.loc[plot_range[0]: plot_range[1]]).show()

In [ ]:
print('same length:', len(segs) == len(segs_averaged))
for feat in feat_names:
    same = (segs[feat].isna() == segs_averaged[feat].isna()).all()
    print(f'NA for {feat}: {same}')

In [ ]:
kwargs = {
    'window': Timedelta(minutes=20),
    'min_valid_duration': Timedelta(minutes=0),
    'agg': 'median',
}
plot_features(rolling_average_seg(segs, **kwargs)).show()

In [ ]:
kwargs = {
    'window': Timedelta(minutes=30),
    'min_valid_duration': Timedelta(minutes=0),
    'agg': 'median',
}
plot_features(rolling_average_seg(segs, **kwargs)).show()

In [ ]:
kwargs = {
    'window': Timedelta(minutes=10),
    'min_valid_duration': Timedelta(minutes=1),
    'agg': 'median',
}
plot_features(rolling_average_seg(segs, **kwargs)).show()

In [ ]:
kwargs = {
    'window': Timedelta(minutes=20),
    'min_valid_duration': Timedelta(minutes=1),
    'agg': 'median',
}
plot_features(rolling_average_seg(segs, **kwargs)).show()

In [ ]:
kwargs = {
    'window': Timedelta(minutes=30),
    'min_valid_duration': Timedelta(minutes=1),
    'agg': 'median',
}
plot_features(rolling_average_seg(segs, **kwargs)).show()

In [ ]:
kwargs = {
    'window': Timedelta(minutes=20),
    'min_valid_duration': Timedelta(minutes=5),
    'agg': 'median',
}
plot_features(rolling_average_seg(segs, **kwargs)).show()

In [ ]:
kwargs = {
    'window': Timedelta(minutes=30),
    'min_valid_duration': Timedelta(minutes=5),
    'agg': 'median',
}
plot_features(rolling_average_seg(segs, **kwargs)).show()

In [ ]:
kwargs = {
    'window': Timedelta(minutes=60),
    'min_valid_duration': Timedelta(minutes=10),
    'agg': 'median',
}
plot_features(rolling_average_seg(segs, **kwargs)).show()

# Behavior of min_periods for rolling average

In [ ]:
# min_periods for index
s = pd.Series([1.0, np.nan, 3.0, 9.0, np.nan, 5.0, 7.0, 8.0], name="x")

out = pd.DataFrame({
    "x": s,
    "median_mp1": s.rolling(window=3, center=True, min_periods=1).median(),
    "median_mp2": s.rolling(window=3, center=True, min_periods=2).median(),
    "median_mp3": s.rolling(window=3, center=True, min_periods=3).median(),
})

print(out)

In [ ]:
import numpy as np
import pandas as pd

# Example data with timestamps (irregular spacing) + NaNs
df = pd.DataFrame({
    "start_mtz": pd.to_datetime([
        "2026-03-26 10:00:00",
        "2026-03-26 10:03:00",
        "2026-03-26 10:06:00",
        "2026-03-26 10:10:00",
        "2026-03-26 10:15:00",
        "2026-03-26 10:19:00",
        "2026-03-26 10:24:00",
    ]),
    "feat": [1.0, np.nan, 3.0, 9.0, np.nan, 5.0, 7.0],
})

out = df.copy()

for mp in [1, 2, 3]:
    out[f"median_mp{mp}"] = (
        df.rolling(
            window=Timedelta(minutes=10),
            on="start_mtz",
            center=True,
            min_periods=mp,
            closed="both",
        )["feat"]
        .median()
    )

print(out)